# 01 - Foods Data And Chunking

Inspect the curated Hue foods knowledge base and the semantic section chunking used by the RAG pipeline. This notebook imports the backend modules, it does not duplicate runtime logic.

Run cells from repo root or from `notebooks/`; the first cell resolves `backend/` from the working directory.

In [ ]:
import sys
from pathlib import Path

for base in (Path.cwd(), *Path.cwd().parents):
    if (base / "backend").is_dir():
        sys.path.insert(0, str(base / "backend"))
        break
print(f"backend on path: {sys.path[0]}")

In [ ]:
from ingestion.chunking.markdown_chunker import _discover_markdown_files

root, files = _discover_markdown_files()
print(f"knowledge base root: {root}")
print(f"markdown files indexed: {len(files)}")
print()
for path in files[:5]:
    print(path.relative_to(root))
print("...")

## Chunk metadata schema

Each chunk carries stable metadata used by ingestion and retrieval:

- `chunk_id`: `{source}|{section}|{index}` - stable per file content
- `source`: KB-relative path, e.g. `foods/restaurants/bun bo hanh.md`
- `title`: document title from the `#` heading
- `section`: semantic section from the `##` heading
- `category`: `foods`
- `subcategory`: folder under `foods/` (`restaurants`, `cafes`, `local_specialties`) or `guide`
- `chunk_type`: `section` (intro content would be `intro`)

No absolute filesystem path is stored.

In [ ]:
from collections import Counter
from ingestion.chunking.markdown_chunker import chunk_foods_markdown

chunks = chunk_foods_markdown()
print(f"total chunks: {len(chunks)}")
print("by subcategory:", dict(Counter(c["metadata"]["subcategory"] for c in chunks)))
print("by chunk_type:", dict(Counter(c["metadata"]["chunk_type"] for c in chunks)))
sizes = sorted(len(c["text"]) for c in chunks)
print(f"text length: min {sizes[0]}, median {sizes[len(sizes)//2]}, max {sizes[-1]}")

## Sample chunks

One chunk per document type: restaurant summary, cafe menu table, local specialty section, and a food guide section.

In [ ]:
samples = {}
for chunk in chunks:
    md = chunk["metadata"]
    key = (md["subcategory"], md["section"])
    if key not in samples:
        samples[key] = chunk
for (sub, section), chunk in sorted(samples.items())[:6]:
    print(f"== {sub} / {section} ==")
    print(chunk["metadata"]["chunk_id"])
    print(chunk["text"][:220])
    print()

In [ ]:
ids = [c["metadata"]["chunk_id"] for c in chunks]
assert len(ids) == len(set(ids)), "chunk_id must be unique"
assert all(c["text"].strip() for c in chunks), "chunk text must be non-empty"
assert all(not c["metadata"]["source"].startswith("/") for c in chunks), "no absolute paths"
assert all(c["metadata"]["section"] != "Nguồn dữ liệu" for c in chunks), "source sections excluded"
assert all("![" not in c["text"] for c in chunks), "image lines stripped"
print("gate checks passed")
print("sample restaurant chunk text:")
print(chunks[0]["text"][:200])

## Edge cases handled

- Image-only lines are stripped from chunk text (URL noise, no caption content).
- Tables stay whole inside a chunk; a table block is never split.
- `## Nguồn dữ liệu` sections are excluded: source tracking is not answer-facing content.
- Long sections (> 1500 chars) are split on blank-line paragraph boundaries; only two food-guides sections exceed the threshold in the current corpus.
- H3 subheadings stay inside their parent H2 section body.

No live model, web, or Qdrant calls happen in this notebook.